# M59f — export-only positional-interleaving repair
Revision: **M59f-2026-09-18-r1**. CPU is sufficient. Run the three code cells in order. No training, dataset setup, upstream clone, or CUDA build.

This replays the trusted M59d and M58 TorchScript artifacts already in Drive and replaces only the positional sine/cosine stack-and-flatten with explicit channel selection. It exports separate internal, layer, and full-model packages. No weights or acceptance thresholds change.

The local Mac run already completed: intermediate and raw-output checks pass; the stricter decoded-candidate gate remains failed. This notebook reproduces export only. **Do not deploy or lower tolerances based on export success.**

In [ ]:
# Cell 1 — all imports, paths, and durable logging. M59f-2026-09-18-r1
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, shlex, shutil, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
REVISION = 'M59f-2026-09-18-r1'
MOBILE_REPO = Path('/content/mobile_adas3d')
MOBILE_URL = 'https://github.com/Ali-RT/mobile_adas3d.git'
ROOT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression')
M59D_DIR = ROOT / 'monodgp_m59d_2d_transformer'
M58_DIR = ROOT / 'monodgp_m58_coreml_conversion'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_ROOT = ROOT / 'monodgp_m59f_position_interleave' / 'runs' / RUN_ID
LOG_DIR = OUTPUT_ROOT / 'colab_logs'
LOG_DIR.mkdir(parents=True, exist_ok=False)
def run_logged(command, cwd, name):
    command = [str(value) for value in command]
    log_path = LOG_DIR / (name + '.log')
    print('+', shlex.join(command), '\nDurable log:', log_path, flush=True)
    tail = deque(maxlen=40)
    with log_path.open('w') as log:
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
            code = process.wait()
    if code: raise RuntimeError(f'Exit {code}; log={log_path}\n' + '\n'.join(tail))
print(REVISION, '\nOutputs:', OUTPUT_ROOT)


In [ ]:
# Cell 2 — own repository on main, dependencies, and required frozen artifacts.
if not MOBILE_REPO.exists():
    run_logged(['git', 'clone', '--branch', 'main', MOBILE_URL, MOBILE_REPO], None, 'clone_mobile')
else:
    branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=MOBILE_REPO, text=True).strip()
    if branch != 'main': raise RuntimeError(f'Expected main; got {branch!r}. Preserve local work before switching.')
    run_logged(['git', 'pull', '--ff-only', 'origin', 'main'], MOBILE_REPO, 'update_mobile')
run_logged([sys.executable, '-m', 'pip', 'install', 'coremltools==9.0', 'numpy>=2.0,<2.4'], MOBILE_REPO, 'dependencies')
SCRIPT = MOBILE_REPO / 'scripts/export_monodgp_m59f_position_interleave.py'
required = [SCRIPT, M59D_DIR/'m59d_2d_transformer_export_gate.json', M59D_DIR/'MonoDGP_M59d_2d_transformer_fp32.pt', M59D_DIR/'m59d_2d_transformer_reference_io.npz', M58_DIR/'m58_coreml_export_gate.json', M58_DIR/'MonoDGP_M58_fixed_fp32.pt', M58_DIR/'m58_reference_io.npz']
missing = [str(path) for path in required if not path.is_file()]
if missing: raise FileNotFoundError('Missing frozen artifacts:\n' + '\n'.join(missing))
print('Project revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=MOBILE_REPO, text=True).strip())


In [ ]:
# Cell 3 — three separate export checks. Mac runtime validation follows.
# Rerun from Cell 1 after failure to preserve earlier evidence in a separate run.
for stage, source in [('internals', M59D_DIR), ('layers', M59D_DIR), ('full', M58_DIR)]:
    destination = OUTPUT_ROOT / stage
    run_logged([sys.executable, '-u', SCRIPT, '--stage', stage, '--source-dir', source, '--output-dir', destination], MOBILE_REPO, 'm59f_' + stage)
    gate = json.loads((destination/'m59f_export_gate.json').read_text())
    assert gate['complete'] and gate['all_export_gates_passed']
    assert not gate['training_performed'] and not gate['weights_changed']
ARCHIVE = shutil.make_archive(str(OUTPUT_ROOT.parent/(RUN_ID + '_m59f')), 'zip', root_dir=OUTPUT_ROOT)
print('Return this archive:', ARCHIVE)
print('Export complete. STOP: this does not mean runtime parity or deployment passed.')


## Mac validation
Use `scripts/validate_monodgp_m59f_macos.py` for each exported stage. For the full model, add `--source-dir` pointing to the original M58 directory to record the unmodified CPU control and exact CPU rewrite equivalence. See `MONODGP_M59F_POSITION_INTERLEAVE_CONTRACT.md` for commands and measured results. Failed parity still writes a diagnostic JSON, then exits nonzero. No Colab rerun is needed for the existing reviewed local results.